In [1]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv("/home/user/MultiAgents/.env")

# 모델 정의
model = init_chat_model(
    "gpt-4o-mini",
    api_key=os.environ["PROXY_TOKEN"],
    base_url=os.environ["PROXY_URL"],  # 핵심: OpenAI SDK의 base_url과 동일 개념
    temperature=0,    
)

# Agentic RAG 구현

## Tool을 활용하여 RAG 구현하기
이 프로젝트는 거대언어모델(LLM)이 하나의 거대한 지식 베이스에서 검색하는 것이 아니라, **질문의 주제에 따라 적절한 문서(규정집)를 스스로 선택하여 검색**하는 **Agentic RAG**를 구현합니다.

### 🛠️ 핵심 메커니즘
1.  **Multi-Retriever**: 주제별(근무시간, 휴가, 복리후생)로 별도의 검색기(Retriever)를 생성합니다.
2.  **Tool Definition**: 각 검색기를 LLM이 사용할 수 있는 '도구(Tool)'로 정의합니다. 이때 함수의 설명(docstring)이 LLM의 판단 기준이 됩니다.
3.  **Agent Execution**: 사용자의 복합적인 질문(예: 휴가와 야근 식대 문의)을 분석하여 필요한 도구들을 순차적으로 호출합니다.

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

/tmp/ipykernel_117149/3589761625.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [3]:
# ==========================================
# 데이터 및 Retriever 준비
# ==========================================
def create_dummy_files():
    data = {
        # 1. 근무시간 규정 (유연근무제 상세 조항 포함)
        "rules_hours.txt": """
제 1장 근무 시간 및 출퇴근 규정

제4조 (소정 근로시간)
1. 회사의 표준 근무시간은 09:00부터 18:00까지로 하며, 점심시간은 12:00부터 13:00까지(1시간)로 한다.
2. 1주간의 근무시간은 휴게시간을 제외하고 40시간을 원칙으로 한다.

제5조 (선택적 근로시간제)
1. 업무 효율 증진을 위해 시차출퇴근제(유연근무제)를 운영한다.
2. 근로자는 08:00~10:00 사이에 30분 단위로 출근 시간을 선택할 수 있다.
   - 08:00 출근 시 -> 17:00 퇴근
   - 09:00 출근 시 -> 18:00 퇴근
   - 10:00 출근 시 -> 19:00 퇴근
3. 단, 부서간 협업을 위해 10:00부터 16:00까지는 필수 근무 시간(Core Time)으로 준수해야 한다.

제6조 (시간외 근무)
1. 연장 근로는 근로기준법에 의거하여 주 12시간을 초과할 수 없다.
2. 야간 및 휴일 근로가 필요한 경우, 사전에 부서장의 승인을 득해야 한다.
""",

        # 2. 휴가 규정 (근속 연수별 차등 및 승인 절차 포함)
        "rules_vacation.txt": """
제 2장 휴가 및 휴직 규정

제11조 (연차 유급휴가)
1. 회사는 1년간 80% 이상 출근한 근로자에게 15일의 유급휴가를 부여한다.
2. 계속 근로 연수가 1년 미만인 근로자 또는 1년간 80% 미만 출근한 근로자에게는 1개월 개근 시 1일의 유급휴가를 부여한다.
3. 연차 휴가는 반일 단위(0.5일)로 사용할 수 있으며, '오전 반차'는 09:00~14:00, '오후 반차'는 14:00~18:00로 한다.

제15조 (장기근속 리프레시 휴가)
1. 회사는 직원의 노고를 치하하고 재충전의 기회를 제공하기 위해 리프레시 휴가 제도를 운영한다.
2. 만 3년 근속 시: 유급 휴가 5일 부여 (발생일로부터 1년 내 사용 원칙)
3. 만 5년 근속 시: 유급 휴가 10일 부여
4. 리프레시 휴가는 연차 휴가와 별도로 부여되며, 미사용 시 수당으로 지급되지 않는다.

제18조 (경조사 휴가)
1. 본인 결혼: 5일
2. 자녀 출산: 배우자 출산 휴가 10일
3. 부모(배우자 부모 포함) 사망: 5일
""",

        # 3. 복리후생 규정 (지급 한도, 증빙 요건 포함)
        "rules_welfare.txt": """
제 3장 복리후생 및 지원 규정

제22조 (식대 지원)
1. 전 임직원에게 월 200,000원의 중식대를 급여와 별도로 지급한다(비과세 적용).
2. 법인카드 사용이 아닌, 급여 지급 시 일괄 지급을 원칙으로 한다.

제23조 (시간외 근무 식대)
1. 야근 식대는 20:00 이후까지 근무한 경우에 한하여 1회 15,000원 한도 내에서 실비 지원한다.
2. 법인카드로 결제 후 그룹웨어 전자결재를 통해 '야근식대' 항목으로 상신해야 한다.
3. 휴일 근무 시에는 중식과 석식 비용을 각각 15,000원 한도 내에서 지원한다.

제25조 (자기개발비 지원)
1. 임직원의 직무 역량 강화를 위해 월 100,000원 한도의 도서 구입비 및 직무 관련 수강료를 지원한다.
2. 매월 말일까지 영수증 및 증빙 자료를 제출하여야 하며, 미사용 이월은 불가하다.
3. 소설, 만화 등 직무와 무관한 도서 구입은 지원 대상에서 제외된다.
"""
    }

    for name, content in data.items():
        with open(name, "w") as f: f.write(content)
        print(f"✅ {name} 파일 생성 완료 ({len(content)}자)")

create_dummy_files()

✅ rules_hours.txt 파일 생성 완료 (495자)
✅ rules_vacation.txt 파일 생성 완료 (509자)
✅ rules_welfare.txt 파일 생성 완료 (466자)


## 1. 데이터 준비 및 Vector Store 생성

RAG(검색 증강 생성)를 위해 텍스트 데이터를 벡터화하여 저장합니다.
여기서는 회사의 사내 규정을 세 가지 카테고리로 나누어 시뮬레이션합니다.

* **Chunking (청킹)**: 긴 문서를 검색하기 좋게 `RecursiveCharacterTextSplitter`로 잘게 쪼갭니다.
* **Embedding (임베딩)**: 쪼개진 텍스트를 `OpenAIEmbeddings`를 사용해 의미를 가진 숫자 벡터로 변환합니다.
* **Vector Store (ChromaDB)**: 변환된 벡터를 빠르게 검색할 수 있는 저장소(ChromaDB)에 담아 `Retriever`로 만듭니다.

In [4]:
# 텍스트 파일을 로드하고 벡터 검색용 retriever를 생성하는 함수를 정의합니다.
def make_retriever(file_path, collection_name):
    # 지정한 경로의 텍스트 파일을 불러오기 위한 로더를 생성합니다.
    loader = TextLoader(file_path)

    # 텍스트 파일 내용을 문서 형태로 로드합니다.
    docs = loader.load()

    # 문서를 일정 길이로 나누기 위한 텍스트 분할기를 생성합니다.
    splitter = RecursiveCharacterTextSplitter(chunk_size=256, chunk_overlap=64)

    # 로드한 문서를 작은 청크 단위로 분할합니다.
    split_docs = splitter.split_documents(docs)
    
    # OpenAI 임베딩 모델을 사용하기 위한 임베딩 객체를 생성합니다.
    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small",  # 사용할 임베딩 모델 이름을 지정합니다.
        api_key=os.environ["PROXY_TOKEN"],  # 환경변수에서 API 키를 불러옵니다.
        base_url=os.environ["PROXY_URL"],   # 프록시 서버의 base_url을 지정합니다.
    )

    # 분할된 문서들을 기반으로 Chroma 벡터 저장소를 생성합니다.
    vector = Chroma.from_documents(
        documents=split_docs,  # 분할된 문서들을 벡터 저장소에 넣습니다.
        embedding=embeddings,  # 문서 임베딩에 사용할 임베딩 객체를 지정합니다.
        collection_name=collection_name  # 주제별 분리를 위해 컬렉션 이름을 지정합니다.
    )

    # 가장 유사한 문서 1개를 반환하는 retriever를 생성해 반환합니다.
    return vector.as_retriever(search_kwargs={'k':1})

In [5]:
# 관련 규정 파일로부터 retriever를 생성합니다.
retriever_hours = make_retriever("/home/user/MultiAgents/Day_1/Ex/rules_hours.txt", "collection_hours")
retriever_vacation = make_retriever("/home/user/MultiAgents/Day_1/Ex/rules_vacation.txt", "collection_vacation")
retriever_welfare = make_retriever("/home/user/MultiAgents/Day_1/Ex/rules_welfare.txt", "collection_welfare")

## 2. Retriever를 에이전트의 도구(Tool)로 변환

단순한 검색기(Retriever)는 LLM이 직접 제어할 수 없습니다. 따라서 이를 **함수 형태의 도구(Tool)** 로 감싸주어야 합니다.

### 💡 핵심 포인트: Docstring (함수 설명)
`@tool` 데코레이터 아래에 작성된 **함수의 docstring(주석)** 은 단순한 설명이 아닙니다.
**Agent(LLM)가 이 도구를 언제 사용해야 할지 판단하는 '프롬프트' 역할**을 합니다.

* `search_working_hours`: 근무 시간 관련 질문일 때 호출됨
* `search_vacation_policy`: 휴가 관련 질문일 때 호출됨
* `search_welfare_benefits`: 복지 관련 질문일 때 호출됨

In [6]:
# ==========================================
# tool 데코레이터로 도구 정의
# ==========================================

# LangChain의 tool 데코레이터를 가져옵니다.
from langchain.tools import tool


# Tool 1: 근무시간 함수
# 근무시간 관련 정보를 검색하는 도구를 정의합니다.
@tool
def search_working_hours(query: str) -> str:
    """
    회사의 근무시간 및 근무 형태에 대한 정보를 검색합니다.

    이 도구는 다음과 같은 질문에 사용됩니다:
    - 출퇴근 시간은 어떻게 되나요?
    - 유연근무제나 시차출퇴근 제도가 있나요?
    - 주 몇 일 근무하나요?
    - 재택근무 또는 원격근무가 가능한가요?
    - 근무시간 관련 규정이나 예외 사항이 있나요?

    입력값(query)은 근무시간, 출퇴근, 근무 형태와 관련된 자연어 질문이어야 합니다.
    """
    # 근무시간 retriever를 사용해 관련 문서를 검색합니다.
    docs = retriever_hours.invoke(query)

    # 검색된 문서들의 본문을 이어붙여 문자열로 반환합니다.
    return "\n\n".join([doc.page_content for doc in docs])


# Tool 2: 휴가 함수
# 휴가 정책 관련 정보를 검색하는 도구를 정의합니다.
@tool
def search_vacation_policy(query: str) -> str:
    """
    회사의 휴가 및 휴무 정책에 대한 정보를 검색합니다.

    이 도구는 다음과 같은 질문에 사용됩니다:
    - 연차는 몇 일 제공되나요?
    - 연차 외에 특별휴가가 있나요?
    - 병가, 경조사 휴가는 어떻게 되나요?
    - 휴가 사용 시 승인 절차가 있나요?
    - 이월 가능한 휴가 규정이 있나요?

    입력값(query)은 연차, 휴가, 병가, 휴무 등 휴가 제도와 관련된 자연어 질문이어야 합니다.
    """
    # 휴가 정책 retriever를 사용해 관련 문서를 검색합니다.
    docs = retriever_vacation.invoke(query)

    # 검색된 문서들의 본문을 이어붙여 문자열로 반환합니다.
    return "\n\n".join([doc.page_content for doc in docs])


# Tool 3: 복리후생 함수
# 복리후생 관련 정보를 검색하는 도구를 정의합니다.
@tool
def search_welfare_benefits(query: str) -> str:
    """
    회사의 복리후생 및 직원 혜택에 대한 정보를 검색합니다.

    이 도구는 다음과 같은 질문에 사용됩니다:
    - 식대, 교통비, 통신비 지원이 있나요?
    - 건강검진이나 보험 혜택이 있나요?
    - 교육비나 자기계발비 지원이 있나요?
    - 사내 복지 시설이나 지원 제도가 있나요?
    - 기타 직원 혜택에는 무엇이 있나요?

    입력값(query)은 복리후생, 지원금, 혜택, 직원 지원 제도와 관련된 자연어 질문이어야 합니다.
    """
    # 복리후생 retriever를 사용해 관련 문서를 검색합니다.
    docs = retriever_welfare.invoke(query)

    # 검색된 문서들의 본문을 이어붙여 문자열로 반환합니다.
    return "\n\n".join([doc.page_content for doc in docs])


# 여러 검색 도구를 하나의 리스트로 묶어 에이전트에 전달할 수 있게 구성합니다.
tools = [search_working_hours, search_vacation_policy, search_welfare_benefits]

/opt/conda/lib/python3.10/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## 3. Agent 생성 및 실행

준비된 도구(Tools)와 LLM(Brain)을 결합하여 에이전트를 생성합니다.

* **Model**: `gpt-4o`를 사용하여 복잡한 추론 능력을 확보합니다.
* **System Prompt**: 에이전트의 페르소나(규정 가이드 봇)와 역할을 부여합니다.
* **Create Agent**: 모델이 사용자의 질문을 분석하고, 앞서 정의한 `tools` 리스트 중에서 적합한 것을 골라 호출(Action)하도록 설정합니다.

### 🚀 실행 시나리오
사용자가 **"3년 근속 리프레시 휴가는 며칠이고, 야근 식대는 얼마인가요?"** 라고 묻는다면 에이전트는 다음과 같이 행동합니다:
1.  질문을 분석하여 '휴가'와 '식대' 두 가지 정보가 필요함을 인지.
2.  `search_vacation_policy` 도구 호출 -> 3년 근속 휴가 정보 획득.
3.  `search_welfare_benefits` 도구 호출 -> 야근 식대 정보 획득.
4.  두 정보를 종합하여 최종 답변 생성.

In [7]:
# ==========================================
# Agent 생성
# ==========================================

# OpenAI 기반 LangChain 채팅 모델 클래스를 가져옵니다.
from langchain_openai import ChatOpenAI

# LangChain v1.0 표준 방식의 create_agent 함수를 가져옵니다.
from langchain.agents import create_agent


# 에이전트가 따라야 할 시스템 프롬프트를 정의합니다.
system_instructions = """
당신은 회사의 사내 규정(근무시간, 휴가, 복리후생)을 안내하는 '규정 가이드 AI'입니다.
사용자의 질문에 답변하기 위해 반드시 제공된 도구(Tool)를 사용하여 정보를 검색해야 합니다.

[업무 지침]
1. **도구 사용 필수**: 사용자의 질문을 분석하여 '근무시간', '휴가', '복리후생' 중 적합한 도구(Tool)를 하나 이상 호출하세요.
2. **복합 질문 처리**: 질문이 여러 카테고리에 걸쳐 있다면(예: "휴가 쓸 때 식대 나오나요?"), 관련된 도구를 모두 호출하여 정보를 종합하세요.
3. **사실 기반 답변**: 오직 도구를 통해 검색된 내용(Context)에 기반해서만 답변하세요. 당신의 사전 지식이나 추측으로 답변하지 마세요.
4. **정보 부재 시**: 검색 결과에 관련 규정이 없다면, "해당 내용은 규정집에 나와있지 않습니다."라고 명확히 말하세요.
5. **답변 태도**: 항상 정중하고 친절한 존댓말(하십시오체 또는 해요체)을 사용하세요.
"""

# 도구와 시스템 프롬프트를 바탕으로 규정 안내용 에이전트를 생성합니다.
agent = create_agent(
    model,
    tools,
    system_prompt=system_instructions
)

In [8]:
# ==========================================
# 실행
# ==========================================

# 사용자 질문 예시를 출력합니다.
print("\n[질문] 3년 근속 리프레시 휴가는 며칠이고, 야근 식대는 얼마인가요?")

# 사용자 질문을 에이전트에 전달하여 답변을 생성합니다.
result = agent.invoke({
    "messages": [("user", "3년 근속 리프레시 휴가는 며칠이고, 야근 식대는 얼마인가요?")]
})

# 응답 메시지 중 마지막 메시지가 최종 답변이므로 이를 출력합니다.
print("\n[답변]")

# 에이전트가 생성한 최종 답변 내용을 출력합니다.
print(result["messages"][-1].content)


[질문] 3년 근속 리프레시 휴가는 며칠이고, 야근 식대는 얼마인가요?

[답변]
3년 근속 시 부여되는 리프레시 휴가는 유급 휴가 5일입니다. 이 휴가는 발생일로부터 1년 내에 사용해야 하며, 연차 휴가와는 별도로 제공됩니다. 미사용 시 수당으로 지급되지 않습니다.

야근 식대는 20:00 이후까지 근무한 경우에 한하여 1회 15,000원 한도 내에서 실비 지원됩니다. 야근 식대를 받기 위해서는 법인카드로 결제한 후, 그룹웨어 전자결재를 통해 '야근식대' 항목으로 상신해야 합니다. 휴일 근무 시에는 중식과 석식 비용을 각각 15,000원 한도 내에서 지원합니다.


In [9]:
# 에이전트 실행 결과 전체를 담고 있는 객체입니다.
result

{'messages': [HumanMessage(content='3년 근속 리프레시 휴가는 며칠이고, 야근 식대는 얼마인가요?', additional_kwargs={}, response_metadata={}, id='5fcd9c64-26a1-44fb-85b5-0a166764c564'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 743, 'total_tokens': 807, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 44, 'engine_ttft_ms': 120, 'engine_ttlt_ms': 2925, 'pre_inference_ms': 132, 'service_tbt_ms': 44, 'service_ttft_ms': 544, 'service_ttlt_ms': 3345, 'total_duration_ms': 3224, 'user_visible_ttft_ms': 413}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_4dcfea0a44', 'id': 'chatcmpl-Dk2sy0g3tMzbvnWfmtcDcHed18c2X', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': 

---
### 🧪 더 해보기 (실습 가이드)

1.  **데이터 수정해보기**: `create_dummy_files` 함수에서 야근 식대를 3만원으로 올리고 다시 실행해보세요. 답변이 바뀌나요?
2.  **새로운 도구 추가하기**: `rules_salary.txt` (연봉 규정)를 만들고 새로운 Tool을 추가하여 에이전트가 연봉 질문에도 답하게 만들어보세요.
3.  **Docstring 변경해보기**: `search_working_hours`의 docstring을 모호하게 바꾸면(예: "문서를 찾습니다") 에이전트가 도구를 잘 선택하는지 관찰해보세요.